# CS229 L01 — Introduction to Machine Learning

**Stanford CS229 · Spring 2026 · Instructor: Tengyu Ma**  
[▶ Lecture Video](https://www.youtube.com/watch?v=DATnpGoGhM8) · [Official Notes](https://cs229.stanford.edu/notes/) · [Course Website](https://cs229.stanford.edu/)

---

**How to use this notebook:**  
> 📌 *Lecture:* — Tengyu Ma's exact words from the lecture transcript  
> 🎯 **Interview:** — Q&A blocks for interview articulation  
LaTeX derivations, code, and external resources fill the gaps.

---

## 1. What is Machine Learning?

> 📌 *Lecture:* "Machine learning is a field of study that gives computers the ability to learn without being explicitly programmed." — Arthur Samuel, 1959

> 📌 *Lecture:* "I found surprisingly some of these definitions still kind of hold — this is 1959, it's like ages ago for machine learning — but somehow the definition still seems to be pretty much the right thing."

**The formal definition (Tom Mitchell, 1998):**

> A computer program is said to **learn** from experience **E** with respect to some class of tasks **T** and performance measure **P** if its performance at tasks T, as measured by P, improves with experience E.

$$\text{ML} = \text{Experience (E)} + \text{Task (T)} + \text{Performance Measure (P)}$$

| Component | Old meaning | 2026 meaning |
|---|---|---|
| **Experience E** | Labeled training data | Data + synthetic data + thinking tokens + web data + human labels |
| **Task T** | One specific task (e.g., chess) | General-purpose: one model, a million tasks |
| **Performance P** | Accuracy on a fixed benchmark | Reward signal, human eval, downstream task metrics |

> 📌 *Lecture:* "These days experiences is even more broad — there are synthetic data you generate, the thinking tokens generated by the models, data from the web, data from the real world, human label data — all of these are kind of experiences."

> 🎯 **Interview:** *How do you define machine learning?*  
> Mitchell's definition captures the key idea: a system learns if it gets better at a task with more experience. In practice, experience = data (labeled or unlabeled, real or synthetic), task = the problem we want to solve, and performance = the metric we care about. What's changed since 1998 is the scale of E and the generality of T — modern LLMs have experience across billions of tokens and generalize across millions of tasks.

## 2. The ML Taxonomy

> 📌 *Lecture:* "I think you have supervised learning, unsupervised learning, and reinforcement learning. Of course they're intersecting — there are a lot of intersections and even more intersections these days."

> 📌 *Lecture:* "Especially these days, I think people mostly think of these as probably methods or paradigms as opposed to the end goal — because in some sense, nobody has the end goal of just doing supervised learning for one task. The end goal is always a general-purpose, automatic learning agent."

```
Machine Learning
├── Supervised Learning     — learn f: X → Y from (x, y) pairs
├── Unsupervised Learning   — find structure in X alone (no Y)
└── Reinforcement Learning  — learn policy from rewards via interaction
```

**How they intersect in 2026:**
- **Pre-training** (LLMs): unsupervised next-token prediction on raw text
- **SFT** (supervised fine-tuning): supervised learning on curated (x, y) pairs
- **RLHF**: reinforcement learning with human reward signal
- **Diffusion models**: technically unsupervised — no labels, just images

> 🎯 **Interview:** *What's the difference between supervised, unsupervised, and reinforcement learning?*  
> Supervised learning has labeled pairs (x, y) and learns a mapping. Unsupervised has only inputs and finds structure — clustering, dimensionality reduction, density estimation. RL has an agent interacting with an environment, taking actions, and receiving rewards — it learns a policy to maximize cumulative reward. In modern ML, these blur: LLM training uses all three — unsupervised pre-training, supervised fine-tuning, and RL with human feedback.

## 3. Supervised Learning

**Setup:** given training data $\{(x^{(i)}, y^{(i)})\}_{i=1}^n$, learn $f: X \to Y$

**Notation used throughout CS229:**
- $x$ — input (features, covariates)
- $y$ — output (label, target, response variable)
- $n$ — number of training examples
- $d$ — input dimension
- $x^{(i)}, y^{(i)}$ — the $i$-th training example

> 📌 *Lecture:* "Often x is used to denote inputs. Y is used to denote output. Here square feet is the input and price is the output and you have many pairs of this because you have many properties."

### 3.1 Regression vs Classification

> 📌 *Lecture:* "If the output is a continuous variable — for example price — then it's called regression. If the output is a discrete variable — for example a label — then it's called classification."

| Problem type | Output $y$ | Example | CS229 lecture |
|---|---|---|---|
| **Regression** | $y \in \mathbb{R}$ | House price given square footage | L02-L03 |
| **Binary classification** | $y \in \{0, 1\}$ | House vs townhouse | L03-L04 |
| **Multiclass classification** | $y \in \{0, ..., K-1\}$ | ImageNet (1000 classes) | L04 |
| **Structured output** | $y$ = bounding box, sequence | Object detection, translation | — |

> 📌 *Lecture:* "All of these large language models fundamentally is a classification problem. When you're predicting the next token or the next word, the word is a discrete label. You have like 50k — these days probably 250k — different words or tokens. You're classifying into all of those discrete choices, right? You have to choose one next word among 50k possible options."

**LLM as classification:**
$$p(\text{next token} \mid x_1, ..., x_t) = \text{softmax}(W h_t) \in \mathbb{R}^{|V|}$$

where $|V|$ is vocabulary size (~50k tokens). This is a 50k-class classification problem at every step.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The core supervised learning setup
np.random.seed(42)
n = 50

# Regression example: house price vs square footage
x_sqft = np.random.uniform(500, 3000, n)           # input: square footage
y_price = 200 * x_sqft + np.random.randn(n) * 50000 + 100000  # output: price

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Regression
axes[0].scatter(x_sqft, y_price / 1e6, alpha=0.6, color='steelblue')
axes[0].set_xlabel('Square footage (x)')
axes[0].set_ylabel('Price in $M (y)')
axes[0].set_title('Regression: y ∈ ℝ')

# Classification example: house vs townhouse
x1 = np.random.randn(n // 2, 2) + [2, 2]   # class 0: houses
x2 = np.random.randn(n // 2, 2) + [-1, -1] # class 1: townhouses
axes[1].scatter(x1[:, 0], x1[:, 1], color='steelblue', label='House (y=0)', marker='o')
axes[1].scatter(x2[:, 0], x2[:, 1], color='coral', label='Townhouse (y=1)', marker='^')
axes[1].set_xlabel('Lot size (x₁)')
axes[1].set_ylabel('Square footage (x₂)')
axes[1].set_title('Classification: y ∈ {0, 1}')
axes[1].legend()

plt.tight_layout()
plt.show()
print("Regression: predict a real number")
print("Classification: predict one of K discrete labels")

### 3.2 The ML Pipeline

> 📌 *Lecture:* "A learning algorithm takes the training data and produces a predictor. Key questions: (1) which predictors are possible? — hypothesis class; (2) how good is a predictor? — loss function; (3) how do we compute the best predictor? — optimization algorithm."

```
Training data {(x⁽ⁱ⁾, y⁽ⁱ⁾)}
        ↓
[1] Hypothesis class    — what functions are possible? (e.g., all linear functions)
[2] Loss function       — how do we measure error? (e.g., squared loss, cross-entropy)
[3] Optimization        — how do we find the best parameters? (e.g., gradient descent)
        ↓
Trained predictor f_θ
        ↓
[4] Evaluation          — how does it perform on held-out test data?
```

This pipeline is the backbone of every CS229 lecture. L02-L03 = steps 1-3 for linear regression. L04 = GLMs (extending hypothesis class). L06 = step 4 (evaluation, bias-variance). L07-L08 = steps 1-3 for neural networks.

> 🎯 **Interview:** *Walk me through the ML development process.*  
> There are four steps: (1) define the hypothesis class — what family of functions can your model represent, e.g., linear functions, neural networks; (2) define the loss function — how you measure how bad a prediction is, e.g., squared loss for regression, cross-entropy for classification; (3) optimize — find the parameters that minimize the training loss, typically with gradient descent; (4) evaluate — measure performance on held-out data, diagnose bias vs variance, iterate. The whole game is making these four choices well for your problem.

## 4. Unsupervised Learning

**Setup:** given data $\{x^{(i)}\}_{i=1}^n$ with **no labels** — find structure.

> 📌 *Lecture:* "Unsupervised learning means that you only have a dataset with only inputs — there's no outputs. There's no clear task because you don't know what they are trying to predict. You just only have inputs... we are trying to get as much as possible from the data without the labels."

### 4.1 Clustering

> 📌 *Lecture:* "There's a technique called clustering which just means that you find data points that are somewhat similar. In 2D, it's actually pretty obvious — if you ask humans to cluster, you probably get similar results."

CS229 covers: K-Means (L09), Gaussian Mixture Models + EM (L09-L10).

### 4.2 Representation Learning / Embeddings

> 📌 *Lecture:* "Every word you can encode it into a numerical vector — sometimes 1000-dimensional. It turns out that these vectors not only have semantical meanings. Similar words have similar vectors. And sometimes the directions correspond to relationships — for example this direction from left to right corresponds to the relationship between capital and country. If you find Italy and don't know the capital, you just go in this direction and find which point you land on."

$$\text{king} - \text{man} + \text{woman} \approx \text{queen}$$
$$\text{Rome} - \text{Italy} + \text{France} \approx \text{Paris}$$

Word2Vec (2013): learn embeddings by predicting surrounding words. This is unsupervised — no human labels, trained on raw Wikipedia text.

CS229 covers: PCA (L10), representation learning (L12).

### 4.3 Generative Models

> 📌 *Lecture:* "Diffusion models — this is about how to generate realistic images from noise or from instructions. This is also unsupervised because you don't have labels for these images. You just have a lot of realistic images and then you can come up with a model such that you can generate more realistic images."

CS229 covers: Diffusion models (L11), VAEs/autoencoders (L12).

> 🎯 **Interview:** *What is unsupervised learning? Give examples.*  
> Unsupervised learning finds structure in unlabeled data. Examples: clustering (K-Means groups similar data points), dimensionality reduction (PCA finds the directions of maximum variance), density estimation (GMMs model the distribution of the data), and representation learning (autoencoders and contrastive methods learn compressed, meaningful representations). Modern LLM pre-training is technically unsupervised — next-token prediction requires no human labels, just raw text.

## 5. Reinforcement Learning

> 📌 *Lecture:* "I think the typical way to think about reinforcement learning is that you are trying to make sequential decisions. At any moment you have to decide what actions you take — how do you move your arm, how do you move the legs. So every time you have to take an action and it's sequential because what actions you take right now affects your future."

**The RL loop:**
$$\text{Agent} \xrightarrow{\text{action } a_t} \text{Environment} \xrightarrow{\text{state } s_{t+1}, \text{ reward } r_t} \text{Agent}$$

**Formal MDP:**  
$(\mathcal{S}, \mathcal{A}, P, R, \gamma)$ — states, actions, transition probabilities, reward function, discount factor

**Goal:** find policy $\pi(a|s)$ that maximizes expected cumulative reward $\mathbb{E}\left[\sum_t \gamma^t r_t\right]$

### 5.1 RL for LLM Training (RLHF)

> 📌 *Lecture:* "These days people also use RL as a tool to train models, especially when the model has stochastic decisions in the middle. You ask a question, it generates a lot of tokens — each time is a stochastic sampling process. It's not a differentiable operation. So to deal with this kind of generation you cannot really backprop through it easily. You have to use reinforcement learning."

> 📌 *Lecture:* "You have some input which is a user prompt. The language model generates text. Then you have another module — either a human or a reward model — that gives a reward indicating whether this is good text or not. You can use this reward as guidance for updating the model to generate better text next round. The buzzwords are RLHF — Reinforcement Learning from Human Feedback — and you can use this to train long-chain thought models."

**Why RL and not supervised learning for RLHF?**  
Because token generation involves stochastic sampling — each token is sampled from a distribution, not computed deterministically. Sampling is not differentiable. You can't backprop through `argmax`. RL (specifically policy gradient) handles this: it optimizes expectations over stochastic policies.

$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta}\left[\sum_t \nabla_\theta \log \pi_\theta(a_t|s_t) \cdot R(\tau)\right]$$

CS229 covers: MDPs, policy gradient, REINFORCE (L15).

### 5.2 Data Collection in RL

> 📌 *Lecture:* "For reinforcement learning, it's a trial-and-error type of algorithm where you iterate between data collection and training. You try some strategy and collect feedbacks. For example you can try how to walk and fall down — there's still data you collected. You collect a trajectory of actions and observations, and those are new data for your training for the next round."

> 📌 *Lecture:* "Automatic theorem proving — you given some statements, you don't have the proofs. First you generate some proofs yourself and you verify and select the correct ones. Those are the new training data. Now you have more training data because you generated them. Then you tune the model on the correct proofs with the hope that after reinforcing, you can generate more correct proofs in the next round. You get more data, tune on more data, keep going — you have a bootstrapping effect."

> 🎯 **Interview:** *How is reinforcement learning used in training large language models?*  
> LLM token generation is stochastic sampling — not differentiable — so standard backprop can't optimize it end-to-end. RL treats each token as an action, the sequence as a trajectory, and uses a reward model (or human feedback) to score the output. The policy gradient theorem lets us compute the gradient of expected reward with respect to model parameters even through the stochastic sampling step. This is the core of RLHF. The key gradient formula is: ∇_θ J = E[∇_θ log π_θ(a|s) · R], which says: actions that led to high reward should be made more probable.

## 6. Course Roadmap

> 📌 *Lecture:* "The methodology of training the models is still very similar to the old days. But the way we are using the models is very different. In the old days you had to get complicated pipelines, get the data, tune the model. Now you just prompt, ask questions directly. But we are more focusing on what's the fundamental technology to tune these models — that's why most of the existing content still applies."

| Block | Lectures | What you build |
|---|---|---|
| Supervised Learning | L01–L06 | Linear regression · logistic regression · GDA · Naive Bayes · bias-variance |
| Neural Networks | L07–L08 | MLP forward pass · backpropagation · training loop |
| Unsupervised | L09–L10 | K-Means · GMM · EM algorithm · PCA |
| Generative + Representation | L11–L12 | Diffusion models · VAEs · contrastive learning · CLIP |
| Language + RL | L13–L15 | LLMs · Transformers · policy gradient · RLHF |

> 📌 *Lecture:* "We're going to have probably one lecture on ML systems — about how to make software and hardware more compatible. If you can make your algorithm run 2x faster you're going to save billions of dollars. And even if it's not saving money — if my algorithm is 2x faster than yours, that's one year for me and two years for you. One year versus two years — that's day and night for these days."

> 🎯 **Interview:** *Why does CS229 still teach linear regression and logistic regression when we have LLMs?*  
> Because these are not outdated techniques — they're the foundation that everything else is built on. Logistic regression is a 1-layer neural network. The cross-entropy loss for a 100-billion parameter LLM is the same loss derived from MLE for logistic regression. Gradient descent for LLMs is the same algorithm as for linear regression, just applied at scale. Understanding the math at the simple case is how you understand what's happening inside the complex case. The methodology is the same; only the scale and architecture differ.

## 7. Key Interview Q&A Summary

> 🎯 **Interview:** *What is the difference between parameters and hyperparameters?*  
> Parameters are the values the model learns from data — weights $w$, bias $b$. Hyperparameters are set before training — learning rate, number of layers, regularization strength $\lambda$. Parameters are optimized by gradient descent. Hyperparameters are chosen by the practitioner, often via a validation set.

> 🎯 **Interview:** *What's the difference between a model and an algorithm?*  
> A model (hypothesis class) defines the family of functions — e.g., all linear functions $w^Tx + b$. An algorithm is the procedure for finding the best function within that family — e.g., gradient descent, normal equations. The model says what's possible; the algorithm says how to search for the best one.

> 🎯 **Interview:** *Why is next-token prediction a classification problem?*  
> The output is one of $|V|$ discrete tokens — at each step, the model produces a probability distribution over the vocabulary via softmax, and the ground truth is a one-hot vector indicating the actual next token. The loss is cross-entropy: $-\log p(y^* | x)$. It's a $|V|$-class classification problem repeated for every token in the sequence.

---

## External Resources

| Resource | What it covers | When to use |
|---|---|---|
| [CS229 Official Notes — Supervised Learning](https://cs229.stanford.edu/notes/cs229-notes1.pdf) | Linear regression derivation, normal equations | Before L02 |
| [Andrew Ng 2018 L01 (YouTube)](https://www.youtube.com/watch?v=jGwO_UgTS7I) | Classic intro — regression, classification framing | Side-by-side with 2026 version |
| [Andrej Karpathy — Intro to ML (30min)](https://www.youtube.com/watch?v=VMj-3S1tku0) | Best code-first ML intro on the internet | After this notebook |
| [The Bitter Lesson — Rich Sutton](http://www.incompleteideas.net/IncIdeas/BitterLesson.html) | Why general methods + compute beat hand-engineering | Context for the ML paradigm shift |
| [Mitchell ML Definition (original paper)](https://www.cs.cmu.edu/~tom/mlbook.html) | The E, T, P framework from source | Historical reference |
| [Lilian Weng — What is RL?](https://lilianweng.github.io/posts/2018-02-19-rl-overview/) | Clean RL taxonomy with MDP formalism | Before L15 |
| [Understanding Deep Learning — Prince Ch. 1](https://udlbook.github.io/udlbook/) | Modern ML intro covering all three paradigms | Companion reading |